In [1]:
EXP1 = "G6sulfur"     # subtrahend  (delta = EXP1 - EXP2)
EXP2 = "SSP245"

In [2]:
import numpy as np
import xarray as xr
import cftime
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import xarray as xr
from pathlib import Path

import sys
import os

analysis_path = os.path.abspath("../20260112_Basic_Analysis")
sys.path.append(analysis_path)

import myfunctions as mf

#Package to suppress Python warnings
import warnings
warnings.filterwarnings("ignore")


import xesmf as xe
import xarray as xr
import numpy as np

# =========================
# Base CEDA paths
# =========================

CEDA_BASE = Path("/badc/cmip6/data/CMIP6")


#Model Names
MODELS = {
    "UKESM1-0-LL":  {"institution": "MOHC",         "ensemble": "r1i1p1f2",  "grid": "gn",},
    "CNRM-ESM2-1":  {"institution": "CNRM-CERFACS", "ensemble": "r1i1p1f2",  "grid": "gr",},
    "MPI-ESM1-2-LR":{"institution": "MPI-M",        "ensemble": "r1i1p1f1",  "grid": "gn",},
    "CESM2-WACCM":  {"institution": "NCAR",         "ensemble": "r1i1p1f1",  "grid": "gn",},
    "IPSL-CM6A-LR": {"institution": "IPSL",         "ensemble": "r1i1p1f1",  "grid": "gr",},
}


# =========================
# Experiment registry
# =========================

# EXP_REGISTRY = {
EXPERIMENTS = {
    # "piControl": {"project": "CMIP",        "scenario": "piControl"},
    "HIST":      {"project": "CMIP",        "scenario": "historical"},
    "SSP245":    {"project": "ScenarioMIP", "scenario": "ssp245"},
    "SSP585":    {"project": "ScenarioMIP", "scenario": "ssp585"},
    "G6solar":   {"project": "GeoMIP",      "scenario": "G6solar"},
    "G6sulfur":  {"project": "GeoMIP",      "scenario": "G6sulfur"},
}

# # Built automatically from EXP1 and EXP2
# EXPERIMENTS = {
#     EXP1: EXP_REGISTRY[EXP1],
#     EXP2: EXP_REGISTRY[EXP2],
# }


In [3]:
# ================================================================
# Utility: year-range label per experiment
# ================================================================

def _exp_year_label(exp_name):
    """Return the year-range suffix for an experiment name."""
    if exp_name in ("G6sulfur", "SSP585", "SSP245"):
        return "20212100"
    elif exp_name == "HIST":
        return "19212000"
    else:
        raise ValueError(f"Unknown experiment '{exp_name}'. "
                         "Add it to _exp_year_label().")

def _time_slice(exp_name):
    """Return the (start, end) time slice strings for an experiment."""
    if exp_name in ("G6sulfur", "SSP585", "SSP245", "G6solar"):
        return ("2021-01-01", "2100-12")   # 80 years
    elif exp_name == "HIST":
        return ("1921-01-01", "2000-12")   # 80 years
    else:
        raise ValueError(f"Unknown experiment '{exp_name}'.")


def roll_lon(da):
    """Roll longitude from 0-360 to -180-180."""
    da = da.assign_coords(lon=(((da.lon + 180) % 360) - 180))
    return da.sortby('lon')


def regrid_to_common(da, target_lat=None, target_lon=None):
    """Regrid a DataArray to common lat/lon grid."""
    if target_lat is None:
        target_lat = np.arange(-50, 51, 2.5)
    elif isinstance(target_lat, tuple):
        target_lat = np.arange(target_lat[0], target_lat[1]+1, 2.5)
    elif isinstance(target_lat, (int, float)):
        target_lat = np.arange(-target_lat, target_lat+1, 2.5)

    if target_lon is None:
        target_lon = np.arange(0, 360, 2.5)  # always default to full lon
    elif isinstance(target_lon, tuple):
        target_lon = np.arange(target_lon[0], target_lon[1]+1, 2.5)

    ds_out = xr.Dataset({
        'lat': (['lat'], target_lat),
        'lon': (['lon'], target_lon)
    })

    # Handle both DataSet and DataArray cases
    if isinstance(da, xr.Dataset):
        ds_in = da
    else:
        ds_in = da.to_dataset(name='data')
        
    regridder = xe.Regridder(ds_in, ds_out, method='bilinear', reuse_weights=False)
    return regridder(da) 
    
def climatology_and_uncertainty(da_year, block_size=80):
    """
    da_year: DataArray with dimension 'year'
    Returns:
        clim_mean  : mean climatology (mean of 30-yr block means)
        clim_sd    : std dev across 30-yr block means
        block_means: DataArray of each 30-yr mean
    """

    # n_years = da_year.sizes["year"]
    n_blocks = 1 #n_years // block_size

    # Trim excess years
    da_trim = da_year.isel(year=slice(0, n_blocks * block_size))

    # Create block index
    block = xr.DataArray(
        np.repeat(np.arange(n_blocks), block_size),
        dims="year",
        coords={"year": da_trim.year},
        name="block"
    )

    # Compute 30-year means
    block_means = (
        da_trim
        .groupby(block)
        .mean(dim="year")*86400
    )

    # Climatological mean (mean of 30-year means)
    clim_mean = block_means.mean(dim="block")

    # Spread across 30-year climatologies
    clim_sd = block_means.std(dim="block")

    return clim_mean, clim_sd, block_means


def load_model_data(base_path, model_name, scenario, mf):
    """
    Generic loader for model data based on model and scenario.

    Parameters
    ----------
    base_path : str or Path
        File path(s) to load
    model_name : str
    scenario : str
    mf : module
        Your module with open_files functions

    Returns
    -------
    xarray.Dataset
    """
    print(str(base_path))

    if model_name == "CESM2-WACCM":
        if scenario == "G6sulfur":
            return mf.open_files_CESM_G6sulfur(base_path)
        elif scenario == "ssp585":
            return mf.open_files_CESM_ssp585(base_path)
        else:
            return mf.open_files(str(base_path))

    elif model_name == "IPSL-CM6A-LR":
        if scenario == "ssp585":
            return mf.open_files_IPSL_ssp585(base_path)
        else:
            return mf.open_files(str(base_path))

    else:
        return mf.open_files(str(base_path))



def load_model_data_alt_path(base_path, project, institution, model_name, scenario, ensemble, grid, mf, var_name=None):
    """
    Generic loader for model data based on model and scenario.
    
    Parameters
    ----------
    base_path : str or Path
        File path(s) to load
    model_name : str
    scenario : str
    mf : module
        Your module with open_files functions
    var_name : str, optional
        Variable name (e.g., 'ua', 'va') to check for downloaded files
    
    Returns
    -------
    xarray.Dataset
    """
    print(str(base_path))
    
    # Check if files exist at base_path, otherwise look in downloaded directory
    base_path = Path(base_path)
    downloaded_base = Path("/home/users/bidyut/data")
    
    # if not base_path.exists() or not list(base_path.glob("*.nc")):
    if not any(base_path.glob("*.nc")):
        # Try downloaded directory
        if var_name:
            # --- special-case ensemble override ---
            if model_name == "CESM2-WACCM":
                ensemble = "r1i1p1f2" if scenario == "G6sulfur" else "r1i1p1f1"
            alt_path = (downloaded_base / project / institution / model_name / scenario / ensemble / "Amon" / 
                       var_name / grid / "latest")
            print(f"Alt path is: {alt_path}")
            if alt_path.exists():
                print(f"  → Using downloaded files at: {alt_path}")
                base_path = alt_path
    
    if model_name == "CESM2-WACCM":
        if scenario == "G6sulfur":
            return mf.open_files_CESM_G6sulfur(base_path)
        elif scenario == "ssp585":
            return mf.open_files_CESM_ssp585(base_path)
        else:
            return mf.open_files(str(base_path))
    elif model_name == "IPSL-CM6A-LR":
        if scenario == "ssp585":
            return mf.open_files_IPSL_ssp585(base_path)
        else:
            return mf.open_files(str(base_path))
    else:
        return mf.open_files(str(base_path))


def roll_and_regrid(varname):
    return roll_lon(regrid_to_common(varname, target_lat=50))



In [4]:
# def climatology(ds):
#     """
#     Calculate seasonal climatologies: JJAS, NDJF, and annual mean.
    
#     Parameters
#     ----------
#     ds : xr.Dataset
#         Input dataset with precipitation data
        
#     Returns
#     -------
#     xr.Dataset
#         Dataset containing climatologies for each season
#     """
#     # Group by month and calculate mean across all years
#     monthly_clim = ds.groupby('time.month').mean('time')
    
#     # JJAS (June-July-August-September): months 6,7,8,9
#     jjas = monthly_clim.sel(month=[6, 7, 8, 9]).mean('month')
    
#     # NDJF (November-December-January-February): months 11,12,1,2
#     ndjf = monthly_clim.sel(month=[11, 12, 1, 2]).mean('month')
    
#     # Annual mean
#     ann = monthly_clim.mean('month')
    
#     # Combine into a single dataset with season dimension
#     clim = xr.concat([jjas, ndjf, ann], dim='season')
#     clim = clim.assign_coords(season=['JJAS', 'NDJF', 'ANN'])
    
#     return clim


def climatology(ds):
    """
    Calculate seasonal climatologies: JJAS, NDJF, and annual mean.
    """
    # Debug: check input
    print(f"  Input ds shape: {ds.dims}")
    print(f"  Input ds variables: {list(ds.data_vars)}")
    print(f"  Sample pr values (first 5): {ds['pr'].values.flat[:5]}")
    
    # JJAS (June-July-August-September): months 6,7,8,9
    jjas_mask = ds['time.month'].isin([6, 7, 8, 9])
    print(f"  JJAS: {jjas_mask.sum().values} timesteps selected")
    jjas = ds.isel(time=jjas_mask).mean('time')
    print(f"  JJAS mean pr sample: {jjas['pr'].values.flat[:5]}")
    
    # NDJF (November-December-January-February): months 11,12,1,2
    ndjf_mask = ds['time.month'].isin([11, 12, 1, 2])
    print(f"  NDJF: {ndjf_mask.sum().values} timesteps selected")
    ndjf = ds.isel(time=ndjf_mask).mean('time')
    
    # Annual mean
    ann = ds.mean('time')
    
    # Combine into a single dataset with season dimension
    clim = xr.concat([jjas, ndjf, ann], dim='season')
    clim = clim.assign_coords(season=['JJAS', 'NDJF', 'ANN'])
    
    return clim

In [5]:
"""
CALCULATE CLIMATOLOGY SECTION
"""

import os

output_file = "./analysis_transient_data/Climatology.nc"

if not os.path.exists(output_file):
    print('#'*40)
    print("File does not exist — running code")
    print('#'*40)


    # ================================================================
    # MAIN PROCESSING LOOP 
    # ================================================================
    all_results = {}
    
    for model_name, model_meta in MODELS.items():
    
        clim_by_exp   = {}      # Store for each experiment
        seasonal_data = {}      # Store seasonal means by year
    
        # ----------------------------
        # 1) LOAD DATA FOR A MODEL : ALL EXPERIMENTS
        # ----------------------------
        for exp, meta in EXPERIMENTS.items():
    
            # --- special-case ensemble override ---
            if model_name == "CESM2-WACCM":
                ensemble = "r1i1p1f2" if meta["scenario"] == "G6sulfur" else "r1i1p1f1"
            else:
                ensemble = model_meta["ensemble"]
    
            def make_base(vname):
                return (
                    CEDA_BASE
                    / meta["project"]
                    / model_meta["institution"]
                    / model_name
                    / meta["scenario"]
                    / ensemble
                    / "Amon"
                    / vname
                    / model_meta["grid"]
                    / "latest"
                )
    
            base_pr = make_base("pr")
            
            # Load (Handle ALTERNATE base paths)
            ds_pr = load_model_data_alt_path(base_pr, meta["project"], model_meta["institution"], 
                                             model_name, meta["scenario"], 
                                             ensemble, model_meta["grid"], mf, "pr")

            #Slice 
            t_start, t_end = _time_slice(exp)
            ds_pr = ds_pr.sel(time=slice(t_start, t_end))
            
            # Roll and regrid
            ds_pr  = roll_and_regrid(ds_pr)
            
            print(f"  [{model_name}] {exp}: pr loaded {ds_pr.sizes['time']} months "
                  f"({ds_pr.time.values[0]} → {ds_pr.time.values[-1]})")
    
            # Calculate climatology
            print(f"  [{model_name}] {exp}: Computing Climatology JJAS, NDJF, ANN")
            clim = climatology(ds_pr)

            # Store for this experiment
            clim_by_exp[exp] = clim
            print(f"  [{model_name}] {exp}: Climatology calculated")
    
        # After all experiments for this model
        all_results[model_name] = clim_by_exp
    
    
    # ================================================================
    # 5) CREATE OUTPUT NETCDF FILE
    # ================================================================
    
    # Stack all models into one dataset
    model_list = []
    for model_name in all_results.keys():
        exp_list = []
        for exp in EXPERIMENTS.keys():
            if exp in all_results[model_name]:
                ds = all_results[model_name][exp]
                ds = ds.expand_dims({'experiment': [exp]})
                exp_list.append(ds)
        
        # Combine experiments for this model
        if exp_list:
            ds_model = xr.concat(exp_list, dim='experiment')
            ds_model = ds_model.expand_dims({'model': [model_name]})
            model_list.append(ds_model)
    
    # Combine all models
    ds_combined = xr.concat(model_list, dim='model')
    
    # Save
    print(f"\nSaving to {output_file}...")
    ds_combined.to_netcdf(output_file)
    print(f"✓ Saved successfully!")
    print(f"  Dimensions: {dict(ds_combined.dims)}")
    print(f"  Variables: {list(ds_combined.data_vars)}")

else:
    print('#'*40)
    print(f"{output_file} already exists — skipping its computation")
    print('#'*40)

########################################
File does not exist — running code
########################################
/badc/cmip6/data/CMIP6/CMIP/MOHC/UKESM1-0-LL/historical/r1i1p1f2/Amon/pr/gn/latest
  [UKESM1-0-LL] HIST: pr loaded 960 months (1921-01-16 00:00:00 → 2000-12-16 00:00:00)
  [UKESM1-0-LL] HIST: Computing Climatology JJAS, NDJF, ANN
  Input ds shape: FrozenMappingWarningOnValuesAccess({'time': 960, 'lat': 41, 'lon': 144})
  Input ds variables: ['pr']
  Sample pr values (first 5): [2.0505528e-05 2.2346769e-05 3.2187290e-05 3.7105889e-05 3.3346761e-05]
  JJAS: 320 timesteps selected
  JJAS mean pr sample: [2.4324083e-05 2.5204488e-05 2.6980624e-05 2.8452614e-05 2.9038438e-05]
  NDJF: 320 timesteps selected
  [UKESM1-0-LL] HIST: Climatology calculated
/badc/cmip6/data/CMIP6/ScenarioMIP/MOHC/UKESM1-0-LL/ssp245/r1i1p1f2/Amon/pr/gn/latest
  [UKESM1-0-LL] SSP245: pr loaded 960 months (2021-01-16 00:00:00 → 2100-12-16 00:00:00)
  [UKESM1-0-LL] SSP245: Computing Climatology JJAS, ND

# END